# Neural Networks Learning the SPY Volatility Surface — Full Notebook + Interactive Extra

Companion notebook to the [Neural Network Vol Surface walkthrough](https://davidariasfinance.com/scripts/neural-network-vol-surface/) on davidariasfinance.com.

This is the **complete runnable version** of the web walkthrough — pull a live SPY option chain from yfinance, build the empirical implied-volatility surface, then train two pure-numpy MLPs (a 49-parameter SMALL net and a 17,409-parameter BIG net) from scratch with real Adam backprop and watch the surface emerge over epochs — plus one **interactive extra** beyond the page:

- **Interactive Plotly 3D surface** — rotate, zoom and hover the BIG net's predicted surface against the real market points.

Because it pulls **live data**, the exact numbers (spot, IV levels) will vary by the day you run it.

---

# PART 1 — The full notebook

The canonical full script, exactly as it appears on the page.

## 1. What we're building

An implied volatility surface is a 2D function. Strike `K` and time to expiry `T` go in, an implied-volatility number comes out. The market quotes one surface for every liquid underlying every minute the exchange is open, and you can pull a snapshot from yfinance in a single API call.

The question this notebook answers: can a small neural network with no built-in knowledge of Black-Scholes, no smile parametrisation, no SVI fit — just a stack of ReLU layers and Adam — **learn** the SPY vol surface from raw quotes? And if so, how much capacity does it actually need?

We train two MLPs side by side. Both see the same data, the same optimiser, the same learning rate, the same five input features. The only difference is the size:

- **SMALL**: 5 -> 4 -> 4 -> 1, **49 parameters**.
- **BIG**: 5 -> 128 -> 128 -> 1, **17,409 parameters**.

Then we snapshot the prediction surface at three checkpoints to watch each one learn in slow motion.

**Heads up.** The results below depend entirely on the data you pull — different days, different liquidity. The point is to demonstrate that the model *can* learn a surface and see how it behaves epoch by epoch. It is not a backtest. There is no train/validation/test split here (that needs a long history of chains, and most free APIs only expose the live snapshot).

## 2. Setup

Five libraries. `yfinance` gives us the option chain. `numpy` handles the MLP forward and backward passes. `pandas` cleans the chain into a tidy `(K, T, IV)` long-format table. `scipy.interpolate.griddata` interpolates that scattered point cloud onto a regular grid for visualization. `matplotlib` renders the surfaces.

In [ ]:
pip install yfinance numpy pandas scipy matplotlib

In [ ]:
import math
from datetime import datetime, timezone

import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from scipy.interpolate import griddata

## 3. Pull a live SPY chain

yfinance exposes the option chain via two calls: `Ticker.options` returns the list of available expiry dates, and `Ticker.option_chain(date)` returns a tuple of `(calls, puts)` dataframes. Each row has a strike, a last price, a bid and ask, volume, open interest, and yfinance's precomputed implied volatility.

We pull the spot price first (today's close), then loop the first 18 expiries. For each contract we keep the rows where the IV looks real (between 5% and 100%), there is some signal of liquidity (volume or open interest), and the last trade was at least a dime. Then we filter to the OTM side at each strike: calls above spot, puts below it. That is the canonical IV-surface convention and avoids the put/call IV divergence that creeps in for ITM contracts.

In [ ]:
# 1. Spot
tk = yf.Ticker("SPY")
S0 = float(tk.history(period="2d", auto_adjust=False)["Close"].iloc[-1])

# 2. Loop the front-of-curve expiries and collect (T, K, side, IV) rows.
rows, now = [], datetime.now(timezone.utc)
for exp_str in tk.options[:18]:
    exp_dt = datetime.strptime(exp_str, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    T = (exp_dt - now).total_seconds() / (365.25 * 86400)
    if not (0.02 < T < 1.5):
        continue
    chain = tk.option_chain(exp_str)
    for side, df in [("call", chain.calls), ("put", chain.puts)]:
        df = df[(df["impliedVolatility"] > 0.05) & (df["impliedVolatility"] < 1.0)]
        df = df[(df["volume"] > 0) | (df["openInterest"] > 0)]
        df = df[df["lastPrice"] > 0.10]
        for _, r in df.iterrows():
            rows.append({"T": T, "K": float(r["strike"]),
                         "side": side, "iv": float(r["impliedVolatility"])})

chain = pd.DataFrame(rows)

# 3. Keep only OTM side at each strike (the canonical IV-surface convention).
chain = chain[((chain["side"] == "call") & (chain["K"] >= S0)) |
              ((chain["side"] == "put")  & (chain["K"] <= S0))]
# 4. Median over duplicates at the same (K, T).
chain = chain.groupby(["K", "T"], as_index=False)["iv"].median()

print(f"Spot S0 = {S0:.2f}   clean (K, T, IV) triplets: {len(chain):,}")

Now we drop the scattered `(K, T, IV)` triplets onto a regular 40x28 grid using `griddata`. Cubic interpolation gives a smooth surface where strikes are dense, and nearest-neighbour fills in the corners where data is sparse. Clip the result to a readable Z range (10% to 45% IV is where SPY actually lives) so the smile structure pops instead of getting flattened by a few outlier wings.

In [ ]:
# Build the visualization grid: K from 85% to 115% of spot, T from 3% to 40% year.
MN_LO, MN_HI = 0.85, 1.15
T_LO,  T_HI  = 0.03, 0.40
N_K, N_T     = 40, 28

K_axis = np.linspace(MN_LO * S0, MN_HI * S0, N_K)
T_axis = np.linspace(T_LO, T_HI, N_T)
K_grid, T_grid = np.meshgrid(K_axis, T_axis)

pts  = chain[["K", "T"]].values
vals = chain["iv"].values

# Cubic where possible, nearest where sparse.
SURFACE = griddata(pts, vals, (K_grid, T_grid), method="cubic")
fill    = griddata(pts, vals, (K_grid, T_grid), method="nearest")
SURFACE = np.where(np.isnan(SURFACE), fill, SURFACE)
SURFACE = np.clip(SURFACE, 0.10, 0.45)

A quick look at the empirical surface — the "truth" the network has to learn. Short-dated OTM puts (front left) trade well above 30% IV (classic equity skew); ATM short-dated options sit closest to 10-15%; as maturity grows the smile flattens toward the long-run vol level.

In [ ]:
Z_LO, Z_HI = 0.10, 0.45
NAVY, ACCENT, GOLD = "#191936", "#323D90", "#C9A24E"

def surface_plot(Z, title, *, truth=None, vmax_clip=0.45):
    """3D surface plot. If truth given, overlay it as a white wireframe."""
    fig = plt.figure(figsize=(7.8, 5.0), dpi=110)
    ax  = fig.add_subplot(111, projection="3d")
    surf = ax.plot_surface(K_grid, T_grid, np.clip(Z, Z_LO, vmax_clip),
                           cmap=cm.viridis, vmin=Z_LO, vmax=vmax_clip,
                           edgecolor="none", alpha=0.92, antialiased=True)
    if truth is not None:
        ax.plot_wireframe(K_grid, T_grid, np.clip(truth, Z_LO, vmax_clip),
                          color="#FFFFFF", alpha=0.45, linewidth=0.6,
                          rstride=4, cstride=4)
    ax.set_xlabel("Strike K", labelpad=6, color=NAVY)
    ax.set_ylabel("T (years)", labelpad=6, color=NAVY)
    ax.set_zlabel("Implied vol", labelpad=6, color=NAVY)
    ax.set_zlim(Z_LO, vmax_clip)
    ax.view_init(elev=22, azim=-55)
    ax.set_title(title, color=NAVY, fontweight="bold", fontsize=12)
    fig.colorbar(surf, ax=ax, shrink=0.55, pad=0.08).set_label("Implied vol", color=NAVY)
    fig.tight_layout(); plt.show()

surface_plot(SURFACE, "SPY implied volatility surface (live yfinance chain)")

## 4. Features

The network input is intentionally minimal. We give it five polynomial features built from moneyness `m = K / S0` and time to expiry `T`:

$$\mathbf{x} = (\, m,\; T,\; m^2,\; T^2,\; m\,T \,)$$

No log-moneyness, no SVI parametrisation, no Black-Scholes structure baked in — just raw polynomial interactions. If the network is going to learn a smile, it has to discover the `m^2 / T` shape on its own.

Then we standardize the features (zero mean, unit variance) and center the IV target around its mean. Both are standard tricks that make gradient descent behave: features on the same scale prevent any single one from dominating the gradient, and a centered target keeps the bias terms from doing all the work in early epochs.

In [ ]:
def features(K, T, S0):
    K = np.asarray(K).flatten()
    T = np.asarray(T).flatten()
    m = K / S0
    return np.stack([m, T, m**2, T**2, m * T], axis=1)

X_full = features(K_grid, T_grid, S0)
y_full = SURFACE.flatten().reshape(-1, 1)

# Standardize features, center the target.
X_mean = X_full.mean(axis=0, keepdims=True)
X_std  = X_full.std(axis=0, keepdims=True) + 1e-9
X_norm = (X_full - X_mean) / X_std

Y_MEAN     = float(y_full.mean())
y_centered = y_full - Y_MEAN

## 5. The MLP, from scratch

No PyTorch, no TensorFlow. Just a plain Python class that stores weights, runs a forward pass with ReLU activations on the hidden layers, and does the backward pass with the **Adam** optimiser — first and second moment estimates of the gradient, bias-corrected for the warmup steps.

Why Adam and not vanilla SGD? Because Adam is the de-facto default for small-data MLP fitting and converges fast enough to show meaningful progress in 600 epochs without tuning the learning rate. The weights are He-initialised ($\sigma = \sqrt{2 / n_{\text{in}}}$) so the variance of the activations does not collapse through the layers.

In [ ]:
class MLP:
    """Feedforward MLP with ReLU activations + Adam optimizer.  Pure numpy."""

    def __init__(self, sizes, seed=42):
        rng = np.random.default_rng(seed)
        self.sizes = sizes
        self.W, self.b = [], []
        self.mW, self.mb, self.vW, self.vb = [], [], [], []
        for i in range(len(sizes) - 1):
            std = math.sqrt(2.0 / sizes[i])                    # He init
            self.W.append(rng.standard_normal((sizes[i], sizes[i + 1])) * std)
            self.b.append(np.zeros((1, sizes[i + 1])))
            self.mW.append(np.zeros_like(self.W[-1]))
            self.mb.append(np.zeros_like(self.b[-1]))
            self.vW.append(np.zeros_like(self.W[-1]))
            self.vb.append(np.zeros_like(self.b[-1]))
        self.t = 0   # Adam timestep counter

    def forward(self, X):
        activations, z_values = [X], []
        a = X
        for i, (W, b) in enumerate(zip(self.W, self.b)):
            z = a @ W + b
            z_values.append(z)
            # ReLU on hidden layers; linear output.
            a = np.maximum(0, z) if i < len(self.W) - 1 else z
            activations.append(a)
        return a, activations, z_values

    def backward(self, y, activations, z_values, lr,
                 beta1=0.9, beta2=0.999, eps=1e-8):
        self.t += 1
        m = y.shape[0]
        delta = 2 * (activations[-1] - y) / m                 # MSE grad
        for i in reversed(range(len(self.W))):
            a_prev = activations[i]
            dW = a_prev.T @ delta
            db = delta.sum(axis=0, keepdims=True)
            if i > 0:
                delta = (delta @ self.W[i].T) * (z_values[i - 1] > 0)
            # Adam moments
            self.mW[i] = beta1 * self.mW[i] + (1 - beta1) * dW
            self.mb[i] = beta1 * self.mb[i] + (1 - beta1) * db
            self.vW[i] = beta2 * self.vW[i] + (1 - beta2) * dW * dW
            self.vb[i] = beta2 * self.vb[i] + (1 - beta2) * db * db
            mW_hat = self.mW[i] / (1 - beta1 ** self.t)
            mb_hat = self.mb[i] / (1 - beta1 ** self.t)
            vW_hat = self.vW[i] / (1 - beta2 ** self.t)
            vb_hat = self.vb[i] / (1 - beta2 ** self.t)
            self.W[i] -= lr * mW_hat / (np.sqrt(vW_hat) + eps)
            self.b[i] -= lr * mb_hat / (np.sqrt(vb_hat) + eps)


def count_params(layers):
    return sum(layers[i] * layers[i + 1] + layers[i + 1]
               for i in range(len(layers) - 1))

## 6. Train both networks

Two architectures, same training loop. 600 epochs, learning rate 0.02. At specific checkpoint epochs we save the current prediction surface so we can visualize how each network is doing.

In [ ]:
SMALL_LAYERS = [5,   4,   4, 1]          #     49 parameters
BIG_LAYERS   = [5, 128, 128, 1]          # 17,409 parameters
EPOCHS       = 600
LR           = 0.02
CHECKPOINTS  = [0, 50, 200, 600]

def train_with_snapshots(layers):
    mlp = MLP(layers, seed=42)
    losses = []
    snaps  = {}
    for epoch in range(EPOCHS + 1):
        y_pred, activs, zs = mlp.forward(X_norm)
        loss = float(np.mean((y_pred - y_centered) ** 2))
        losses.append(loss)
        if epoch in CHECKPOINTS:
            snaps[epoch] = (y_pred.flatten() + Y_MEAN).reshape(SURFACE.shape)
        if epoch < EPOCHS:
            mlp.backward(y_centered, activs, zs, LR)
    return mlp, snaps, np.array(losses)

small_net, small_snaps, small_losses = train_with_snapshots(SMALL_LAYERS)
big_net,   big_snaps,   big_losses   = train_with_snapshots(BIG_LAYERS)

print(f"SMALL ({count_params(SMALL_LAYERS):>5} params)  final RMSE = "
      f"{math.sqrt(small_losses[-1]) * 100:.2f}% IV")
print(f"BIG   ({count_params(BIG_LAYERS):>5} params)  final RMSE = "
      f"{math.sqrt(big_losses[-1]) * 100:.2f}% IV")

## 7. Watch them learn

Same plotting routine, applied to each saved snapshot. The white wireframe overlay on each chart is the true SPY surface (the "target"), and the colored surface underneath is the network's current prediction.

- **Epoch 50** — both nets have caught the average level (~18-22% IV). The BIG net is starting to bend; the SMALL net is still mostly a plane.
- **Epoch 200** — both now have a smile shape. The BIG net is close to the truth including the steep OTM-put corner; the SMALL net has the right shape but lacks the resolution to capture the corner curl.
- **Epoch 600** — converged. The BIG net is essentially indistinguishable from the truth; the SMALL net stays blunt at the deep OTM-put corner. It does not have enough hidden units to bend that sharply.

In [ ]:
for ep in CHECKPOINTS:
    surface_plot(small_snaps[ep],
                 f"SMALL net (5-4-4-1, 49 params) - epoch {ep}", truth=SURFACE)
    surface_plot(big_snaps[ep],
                 f"BIG net (5-128-128-1, 17,409 params) - epoch {ep}", truth=SURFACE)

## 8. Training loss. Capacity matters.

The story of those snapshots in one chart. Y axis is training RMSE on a log scale (epoch 0 starts near 100% and converges under 1%). Both networks crash through the same trajectory for the first ~80 epochs, learning the smile's mean level and rough skew. After that they separate: the BIG net keeps shaving off error well past epoch 300, the SMALL net flattens. The gap is small in absolute terms, but it is exactly the corner of the surface that matters most for OTM-put pricing.

In [ ]:
fig, ax = plt.subplots(figsize=(7.8, 3.4), dpi=110)
xs = np.arange(1, EPOCHS + 1)   # skip the wildly random epoch 0
ax.plot(xs, np.sqrt(small_losses[1:]) * 100, color="#C0392B", linewidth=2.2,
        label=f"SMALL (5-4-4-1, {count_params(SMALL_LAYERS)} params)")
ax.plot(xs, np.sqrt(big_losses[1:]) * 100, color=ACCENT, linewidth=2.2,
        label=f"BIG (5-128-128-1, {count_params(BIG_LAYERS):,} params)")
ax.set_xlabel("Epoch")
ax.set_ylabel("Training RMSE (% IV)  -  log scale")
ax.set_yscale("log")
ax.set_title("Training error per epoch  -  capacity matters",
             color=NAVY, fontweight="bold")
ax.set_xlim(1, EPOCHS)
ax.grid(True, alpha=0.25, linestyle=":")
ax.legend(loc="upper right", frameon=True)
fig.tight_layout(); plt.show()

## 9. Recap

- A volatility surface is a 2D function of strike and expiry. Neural networks can learn it from raw quotes — no Black-Scholes structure required.
- Five polynomial features (`m`, `T`, `m^2`, `T^2`, `m*T`) are enough input.
- A pure-numpy MLP with Adam works fine. No need for PyTorch unless you want autograd for free.
- Capacity decides what gets captured. A 49-parameter net gets the mean level and rough skew but blunts the OTM-put corner; a 17k-parameter net captures it precisely.
- The same code generalises to any underlying with a chain on yfinance — swap `SPY` for `TLT`, `QQQ`, `NVDA`, anything liquid.

---

# EXTRA — Interactive Plotly 3D surface

The matplotlib renders above are static. Here we rebuild the **BIG net's** predicted surface as an interactive Plotly 3D plot — drag to rotate, scroll to zoom, hover any point for its exact `(K, T, IV)` value — and overlay the **real market quotes** as a scatter so you can see exactly where the network agrees with, and departs from, the data it learned.

Requires Plotly: `pip install plotly`.

In [ ]:
try:
    import plotly.graph_objects as go
except ImportError:
    print("Install plotly:  pip install plotly")
else:
    # Reuse the trained BIG net + the Part-1 grid. Predict over the full grid.
    big_pred, _, _ = big_net.forward(X_norm)
    BIG_SURFACE = (big_pred.flatten() + Y_MEAN).reshape(SURFACE.shape)
    BIG_SURFACE = np.clip(BIG_SURFACE, Z_LO, Z_HI)

    fig = go.Figure()

    # 1. BIG net predicted surface.
    fig.add_trace(go.Surface(
        x=K_axis, y=T_axis, z=BIG_SURFACE,
        colorscale="Viridis", cmin=Z_LO, cmax=Z_HI, opacity=0.92,
        colorbar=dict(title="Implied vol"),
        name="BIG net surface", showscale=True,
    ))

    # 2. Real market quotes as scatter (the data the net learned from).
    fig.add_trace(go.Scatter3d(
        x=chain["K"].values, y=chain["T"].values, z=chain["iv"].values,
        mode="markers",
        marker=dict(size=3, color="#C9A24E", line=dict(width=0)),
        name="Real market quotes",
        hovertemplate="K=%{x:.0f}<br>T=%{y:.3f}<br>IV=%{z:.1%}<extra></extra>",
    ))

    fig.update_layout(
        title="BIG net predicted SPY vol surface vs real market quotes (interactive)",
        scene=dict(
            xaxis_title="Strike K",
            yaxis_title="T (years)",
            zaxis_title="Implied vol",
            camera=dict(eye=dict(x=1.6, y=-1.6, z=0.9)),
        ),
        width=920, height=640,
        legend=dict(x=0.0, y=0.95),
    )
    fig.show()

---

That's the full notebook plus both extras. Pair it with the [Heston Vol Surface notebook](https://davidariasfinance.com/scripts/heston-vol-surface/) for the parametric, financial-math approach to the same problem — Heston says *the smile must come from five SDE parameters*; the neural net says *give me the data*. Both end up fitting the market; the choice is interpretability vs. flexibility.

Reply to the welcome email if anything breaks.

Built and modelled by **David Arias, CFA** — [davidariasfinance.com](https://davidariasfinance.com)
Source page: [Neural Networks Learning the SPY Vol Surface](https://davidariasfinance.com/scripts/neural-network-vol-surface/)

— David